# 05. Application + платежи по рассрочкам

## Цель

Кратко:
- создать признаки из `installments_payments.csv`;
- добавить только платёжные признаки к `application_train`;
- обучить CatBoost и записать `application_installments`.


## 1. Импорты и пути


In [1]:
from pathlib import Path
import sys


def _is_project_root(path):
    return (
        (path / "src").is_dir()
        and (path / "notebooks").is_dir()
        and (path / "data").is_dir()
    )


project_candidates = [
    Path.cwd(),
    *Path.cwd().parents,
    Path("/content/credit-scoring-system"),
]

if "google.colab" in sys.modules:
    from google.colab import drive

    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.is_dir():
        drive.mount("/content/drive")

    default_drive_project = (
        drive_root / "credit-scoring-system"
    )
    project_candidates.append(default_drive_project)

    if not any(
        _is_project_root(path)
        for path in project_candidates
    ):
        project_candidates.extend(
            config_path.parents[1]
            for config_path in drive_root.rglob("src/config.py")
        )

PROJECT_ROOT = next(
    (
        path.resolve()
        for path in project_candidates
        if _is_project_root(path)
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не найден корень credit-scoring-system. На Google Drive "
        "должна находиться вся папка проекта с src/, notebooks/ и data/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_setup import setup_notebook


PROJECT_ROOT = setup_notebook()

Mounted at /content/drive
Installing missing dependency: catboost
Environment: Google Colab
Python: 3.12.13
Project root: /content/drive/MyDrive/credit-scoring-system
Raw data: /content/drive/MyDrive/credit-scoring-system/data/raw
Models: /content/drive/MyDrive/credit-scoring-system/models
Reports: /content/drive/MyDrive/credit-scoring-system/reports


In [2]:
import numpy as np
import pandas as pd
from catboost import (
    CatBoostClassifier,
    Pool,
    cv as catboost_cv,
)
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

from src.config import (
    INTERIM_DATA_DIR as DATA_INTERIM_DIR,
    PROCESSED_DATA_DIR as DATA_PROCESSED_DIR,
    find_data_file,
)
from src.experiment_tracking import save_experiment_result
from src.model_config import (
    get_catboost_device_config,
    get_catboost_gpu_count,
    print_catboost_device_info,
)


APPLICATION_PATH = find_data_file("application_train.csv")
INSTALLMENTS_PATH = find_data_file("installments_payments.csv")
CLIENT_SPLIT_PATH = (
    DATA_PROCESSED_DIR / "client_split.csv"
)
INSTALLMENTS_FEATURES_PATH = (
    DATA_INTERIM_DIR / "installments_features.csv"
)
RANDOM_STATE = 42
CV_FOLDS = 3


### Устройство CatBoost


In [3]:
gpu_count = get_catboost_gpu_count()
catboost_device_config = get_catboost_device_config(
    gpu_count=gpu_count,
)
print_catboost_device_info(
    catboost_device_config,
    gpu_count=gpu_count,
)


Modeling environment: Google Colab
CatBoost GPU count: 1
CatBoost device: GPU
CatBoost GPU devices: 0


## 2. Загрузка application_train

Основная таблица содержит одну строку на клиента. Техническое значение
`365243` в `DAYS_EMPLOYED` заменяется пропуском.


In [4]:
application = pd.read_csv(APPLICATION_PATH)

if "DAYS_EMPLOYED" in application.columns:
    application["DAYS_EMPLOYED"] = application[
        "DAYS_EMPLOYED"
    ].replace(365243, np.nan)

assert application["SK_ID_CURR"].is_unique
assert application["TARGET"].isin([0, 1]).all()

print("Application:", application.shape)
display(application.head())


Application: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Загрузка installments_payments


In [5]:
installments = pd.read_csv(INSTALLMENTS_PATH)

assert {"SK_ID_CURR", "SK_ID_PREV"}.issubset(
    installments.columns
)

print("Installments:", installments.shape)
display(installments.head())


Installments: (13605401, 8)


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


## 4. Признаки на уровне платежа

Будущие и не совершённые платежи исключаются. Просрочка измеряется только
положительным числом дней, а нулевой плановый платёж не используется как
знаменатель.


In [6]:
known_payment = (
    installments["DAYS_INSTALMENT"].le(0)
    & installments["DAYS_ENTRY_PAYMENT"].notna()
    & installments["DAYS_ENTRY_PAYMENT"].le(0)
)

installments = installments[
    known_payment
].copy()

installments["PAYMENT_DELAY_DAYS"] = (
    installments["DAYS_ENTRY_PAYMENT"]
    - installments["DAYS_INSTALMENT"]
).clip(lower=0)

installments["IS_LATE"] = (
    installments["PAYMENT_DELAY_DAYS"].gt(0)
).astype(int)

installments["IS_UNDERPAID"] = (
    installments["AMT_PAYMENT"]
    < installments["AMT_INSTALMENT"]
).astype(int)


## 5. Агрегация до клиента


In [7]:
installments_features = (
    installments
    .groupby("SK_ID_CURR")
    .agg(
        INST_PAYMENT_COUNT=("SK_ID_PREV", "size"),
        INST_PLANNED_AMOUNT_TOTAL=("AMT_INSTALMENT", "sum"),
        INST_PAID_AMOUNT_TOTAL=("AMT_PAYMENT", "sum"),
        INST_PAYMENT_AMOUNT_MEAN=("AMT_PAYMENT", "mean"),
        INST_UNDERPAID_SHARE=("IS_UNDERPAID", "mean"),
        INST_LATE_PAYMENT_SHARE=("IS_LATE", "mean"),
        INST_DELAY_DAYS_MEAN=("PAYMENT_DELAY_DAYS", "mean"),
        INST_DELAY_DAYS_MAX=("PAYMENT_DELAY_DAYS", "max"),
    )
    .reset_index()
)

installments_features["INST_PAID_TO_PLANNED_RATIO"] = (
    installments_features["INST_PAID_AMOUNT_TOTAL"]
    / installments_features[
        "INST_PLANNED_AMOUNT_TOTAL"
    ].replace(0, np.nan)
)

assert installments_features["SK_ID_CURR"].is_unique

print(installments_features.shape)
display(installments_features.head())


(339578, 10)


,SK_ID_CURR,INST_PAYMENT_COUNT,INST_PLANNED_AMOUNT_TOTAL,INST_PAID_AMOUNT_TOTAL,INST_PAYMENT_AMOUNT_MEAN,INST_UNDERPAID_SHARE,INST_LATE_PAYMENT_SHARE,INST_DELAY_DAYS_MEAN,INST_DELAY_DAYS_MAX,INST_PAID_TO_PLANNED_RATIO
0,100001,7,41195.925,41195.925,5885.132143,0.0,0.142857,1.571429,11.0,1.0
1,100002,19,219625.695,219625.695,11559.247105,0.0,0.000000,0.000000,0.0,1.0
2,100003,25,1618864.650,1618864.650,64754.586000,0.0,0.000000,0.000000,0.0,1.0
3,100004,3,21288.465,21288.465,7096.155000,0.0,0.000000,0.000000,0.0,1.0
4,100005,9,56161.845,56161.845,6240.205000,0.0,0.111111,0.111111,1.0,1.0


## 6. Проверка и сохранение признаков


In [8]:
assert installments_features["SK_ID_CURR"].is_unique
assert "TARGET" not in installments_features.columns

installments_features.to_csv(
    INSTALLMENTS_FEATURES_PATH,
    index=False,
)

print("Сохранено:", INSTALLMENTS_FEATURES_PATH)


Сохранено: /content/drive/MyDrive/credit-scoring-system/data/interim/installments_features.csv


## 7. Merge с application


In [9]:
modeling_data = application.merge(
    installments_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)

assert len(modeling_data) == len(application)
assert modeling_data["SK_ID_CURR"].is_unique

print("Application:", application.shape)
print("После добавления installments:", modeling_data.shape)
print(
    "Добавлено признаков:",
    modeling_data.shape[1] - application.shape[1],
)


Application: (307511, 122)
После добавления installments: (307511, 131)
Добавлено признаков: 9


## Чтение единого client split


In [10]:
if not CLIENT_SPLIT_PATH.exists():
    raise FileNotFoundError(
        "Сначала выполните notebooks/02_application_baseline.ipynb. "
        f"Ожидаемый файл: {CLIENT_SPLIT_PATH}"
    )

client_split = pd.read_csv(CLIENT_SPLIT_PATH)

assert client_split.columns.tolist() == ["SK_ID_CURR", "split"]
assert client_split["SK_ID_CURR"].is_unique
assert set(client_split["split"]) == {"train", "holdout"}
assert set(client_split["SK_ID_CURR"]) == set(application["SK_ID_CURR"])

modeling_data = modeling_data.merge(
    client_split,
    on="SK_ID_CURR",
    how="inner",
    validate="one_to_one",
)

assert len(modeling_data) == len(application)
assert modeling_data["SK_ID_CURR"].is_unique

print(client_split["split"].value_counts())


split
train      246008
holdout     61503
Name: count, dtype: int64


## 9. Создание X и y

`TARGET`, идентификатор клиента и техническая колонка разделения не
передаются модели.


In [11]:
train_data = modeling_data[
    modeling_data["split"].eq("train")
].copy()

n_holdout = int(
    modeling_data["split"].eq("holdout").sum()
)

feature_columns = [
    column
    for column in modeling_data.columns
    if column not in {
        "TARGET",
        "SK_ID_CURR",
        "split",
    }
]

X_train = train_data[feature_columns]
y_train = train_data["TARGET"].astype(int)

assert "TARGET" not in X_train.columns
assert "SK_ID_CURR" not in X_train.columns
assert "split" not in X_train.columns

print("Train:", X_train.shape)
print("Holdout clients (не используется):", n_holdout)


Train: (246008, 129)
Holdout clients (не используется): 61503


## 10. Подготовка данных для CatBoost

Категориальные пропуски заменяются строкой. Числовые `NaN` остаются без
изменений: CatBoost обрабатывает их самостоятельно.


In [12]:
categorical_columns = (
    X_train
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

X_train_catboost = X_train.copy()
X_train_catboost[categorical_columns] = (
    X_train_catboost[categorical_columns]
    .fillna("Unknown")
    .astype(str)
)

print("Категориальных признаков:", len(categorical_columns))


Категориальных признаков: 16


## 11. Стратифицированная кросс-валидация


In [13]:
cv_splitter = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


## 12. Pool для CatBoost


In [14]:
train_pool = Pool(
    data=X_train_catboost,
    label=y_train,
    cat_features=categorical_columns,
)


## 13. Параметры CatBoost


In [15]:
catboost_params = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "custom_metric": ["PRAUC:type=Classic"],
    "auto_class_weights": "Balanced",
    "random_seed": RANDOM_STATE,
    "allow_writing_files": False,
    "verbose": False,
    **catboost_device_config,
}


## 14. Библиотечная CV CatBoost

OOF-предсказания не требуются, поэтому используется `catboost.cv()`
без ручного цикла по фолдам. Test в CV не участвует.


In [16]:
catboost_cv_results = catboost_cv(
    pool=train_pool,
    params=catboost_params,
    folds=cv_splitter,
    early_stopping_rounds=100,
    as_pandas=True,
    verbose=100,
)


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:877: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(
Default metric period is 5 because AUC, PRAUC is/are not implemented for GPU


Training on fold [0/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7009606	best: 0.7009606 (0)	total: 166ms	remaining: 2m 46s
100:	test: 0.7484168	best: 0.7484168 (100)	total: 10.3s	remaining: 1m 31s
200:	test: 0.7512544	best: 0.7512562 (197)	total: 19.2s	remaining: 1m 16s
300:	test: 0.7538913	best: 0.7538981 (294)	total: 28.2s	remaining: 1m 5s
400:	test: 0.7554616	best: 0.7554616 (396)	total: 37.8s	remaining: 56.4s
500:	test: 0.7569781	best: 0.7569781 (500)	total: 45.8s	remaining: 45.6s
600:	test: 0.7578604	best: 0.7578712 (596)	total: 55.5s	remaining: 36.9s
700:	test: 0.7585935	best: 0.7585962 (697)	total: 1m 5s	remaining: 27.9s
800:	test: 0.7590250	best: 0.7590250 (800)	total: 1m 13s	remaining: 18.2s
900:	test: 0.7594528	best: 0.7594727 (889)	total: 1m 23s	remaining: 9.21s
999:	test: 0.7596776	best: 0.7597013 (998)	total: 1m 33s	remaining: 0us
bestTest = 0.7597013116
bestIteration = 998
Training on fold [1/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7061496	best: 0.7061496 (0)	total: 166ms	remaining: 2m 45s
100:	test: 0.7487363	best: 0.7487363 (100)	total: 9.69s	remaining: 1m 26s
200:	test: 0.7523246	best: 0.7523434 (197)	total: 18.8s	remaining: 1m 14s
300:	test: 0.7542329	best: 0.7542499 (293)	total: 28.5s	remaining: 1m 6s
400:	test: 0.7560148	best: 0.7560148 (397)	total: 36.5s	remaining: 54.5s
500:	test: 0.7570800	best: 0.7571153 (494)	total: 46s	remaining: 45.9s
600:	test: 0.7577475	best: 0.7577479 (598)	total: 55.6s	remaining: 36.9s
700:	test: 0.7589794	best: 0.7589974 (692)	total: 1m 6s	remaining: 28.3s
800:	test: 0.7596616	best: 0.7596616 (800)	total: 1m 15s	remaining: 18.9s
900:	test: 0.7601115	best: 0.7601343 (895)	total: 1m 25s	remaining: 9.41s
999:	test: 0.7605338	best: 0.7605338 (999)	total: 1m 33s	remaining: 0us
bestTest = 0.7605337501
bestIteration = 999
Training on fold [2/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7054772	best: 0.7054772 (0)	total: 225ms	remaining: 3m 44s
100:	test: 0.7487031	best: 0.7487031 (100)	total: 9.3s	remaining: 1m 22s
200:	test: 0.7507197	best: 0.7507197 (200)	total: 19.1s	remaining: 1m 15s
300:	test: 0.7524957	best: 0.7524957 (299)	total: 28.4s	remaining: 1m 5s
400:	test: 0.7549203	best: 0.7549203 (398)	total: 36.9s	remaining: 55.1s
500:	test: 0.7570242	best: 0.7570242 (497)	total: 46.6s	remaining: 46.4s
600:	test: 0.7580985	best: 0.7580985 (600)	total: 55.4s	remaining: 36.8s
700:	test: 0.7585289	best: 0.7585508 (699)	total: 1m 4s	remaining: 27.5s
800:	test: 0.7591793	best: 0.7591960 (779)	total: 1m 14s	remaining: 18.5s
900:	test: 0.7594897	best: 0.7594999 (893)	total: 1m 22s	remaining: 9.09s
999:	test: 0.7596610	best: 0.7596624 (990)	total: 1m 32s	remaining: 0us
bestTest = 0.7596624494
bestIteration = 990


## 15. Лучшая итерация и CV-метрики


In [17]:
auc_mean_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-mean")
)

auc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-std")
)

pr_auc_mean_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-mean")
)

pr_auc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-std")
)

best_cv_index = catboost_cv_results[
    auc_mean_column
].idxmax()

best_cv_row = catboost_cv_results.loc[
    best_cv_index
]

best_iteration = int(
    best_cv_row["iterations"]
) + 1

cv_roc_auc = float(
    best_cv_row[auc_mean_column]
)

cv_roc_auc_std = float(
    best_cv_row[auc_std_column]
)

cv_pr_auc = float(
    best_cv_row[pr_auc_mean_column]
)

cv_pr_auc_std = float(
    best_cv_row[pr_auc_std_column]
)

print(f"Лучшая итерация: {best_iteration}")
print(
    f"CV ROC-AUC: {cv_roc_auc:.4f} "
    f"± {cv_roc_auc_std:.4f}"
)
print(
    f"CV PR-AUC: {cv_pr_auc:.4f} "
    f"± {cv_pr_auc_std:.4f}"
)


Лучшая итерация: 1000
CV ROC-AUC: 0.7600 ± 0.0005
CV PR-AUC: nan ± nan


## 16. Итоговая модель CatBoost


In [18]:
catboost_model = CatBoostClassifier(
    iterations=best_iteration,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    verbose=100,
    **catboost_device_config,
)


## 17. Обучение итоговой модели


In [19]:
catboost_model.fit(
    X_train_catboost,
    y_train,
    cat_features=categorical_columns,
)


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 62.3ms	remaining: 1m 2s
100:	total: 6.06s	remaining: 54s
200:	total: 10.4s	remaining: 41.2s
300:	total: 14.7s	remaining: 34.1s
400:	total: 20.5s	remaining: 30.6s
500:	total: 24.8s	remaining: 24.7s
600:	total: 29.1s	remaining: 19.3s
700:	total: 34.8s	remaining: 14.8s
800:	total: 39.1s	remaining: 9.72s
900:	total: 43.6s	remaining: 4.79s
999:	total: 49.2s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', depth=6, devices='0', eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

## Запись CV-результата


In [20]:
current_result = {
    "experiment": "application_installments",
    "notebook": "05_installments_features.ipynb",
    "model": "CatBoostClassifier",
    "feature_set": "application + installments",
    "source_tables": "application_train.csv, installments_payments.csv",
    "device": catboost_device_config["task_type"],
    "n_train": len(train_data),
    "n_holdout": n_holdout,
    "n_features": X_train.shape[1],
    "cv_folds": CV_FOLDS,
    "best_iteration": best_iteration,
    "cv_roc_auc": cv_roc_auc,
    "cv_roc_auc_std": cv_roc_auc_std,
    "cv_pr_auc": cv_pr_auc,
    "cv_pr_auc_std": cv_pr_auc_std,
    "holdout_roc_auc": None,
    "holdout_pr_auc": None,
}

all_results = save_experiment_result(current_result)
display(all_results)


,experiment,notebook,model,feature_set,source_tables,device,n_train,n_holdout,n_features,cv_folds,best_iteration,cv_roc_auc,cv_roc_auc_std,cv_pr_auc,cv_pr_auc_std,holdout_roc_auc,holdout_pr_auc
0,application_logistic,02_application_baseline.ipynb,LogisticRegression,application,application_train.csv,CPU,246008,61503,120,3,NaN,0.744841,0.002434,0.217865,0.005206,NaN,NaN
1,application_catboost,02_application_baseline.ipynb,CatBoostClassifier,application,application_train.csv,GPU,246008,61503,120,3,1000.0,0.754176,0.002317,NaN,NaN,NaN,NaN
2,application_bureau,03_bureau_features.ipynb,CatBoostClassifier,application + bureau,"application_train.csv, bureau.csv, bureau_bala...",GPU,246008,61503,134,3,1000.0,0.758362,0.001620,NaN,NaN,NaN,NaN
3,application_previous,04_previous_application_features.ipynb,CatBoostClassifier,application + previous_application,"application_train.csv, previous_application.csv",GPU,246008,61503,131,3,1000.0,0.760286,0.002017,NaN,NaN,NaN,NaN
4,application_installments,05_installments_features.ipynb,CatBoostClassifier,application + installments,"application_train.csv, installments_payments.csv",GPU,246008,61503,129,3,1000.0,0.759957,0.000499,NaN,NaN,NaN,NaN


## Выводы


In [21]:
print("Эксперимент: application + installments_payments")
print(f"Количество признаков: {X_train.shape[1]}")
print(f"Лучшая итерация: {best_iteration}")
print(f"CV ROC-AUC: {cv_roc_auc:.4f}")
print(f"CV PR-AUC: {cv_pr_auc:.4f}")
print("Holdout не использовался: результат сравнивается только по CV.")


Эксперимент: application + installments_payments
Количество признаков: 129
Лучшая итерация: 1000
CV ROC-AUC: 0.7600
CV PR-AUC: nan
Holdout не использовался: результат сравнивается только по CV.
